In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os 
from src.utils import get_args, load_config, set_token, get_files
from src.doc_utils import process_file
from src.text_preprocessing import get_chunks
from src.models import Chatbot
from src.app_utils import generate_sidebar

from PyPDF2 import PdfReader
import yaml

Could not import azure.core python package.


In [3]:
config = load_config('config.yaml')

In [4]:
with open('secrets.yaml', "r") as stream:
    secrets = yaml.safe_load(stream)
api_token = secrets['HUGGINGFACEHUB_API_TOKEN']

In [5]:
uploaded_file = ['data/1706.03762.pdf']
model_repo, embeddings_model_name, model_name, model_kwargs, api_token, run_local = 'huggingface', 'google/flan-t5-base', 'google/flan-t5-base', {}, api_token, False
os.environ["HUGGINGFACEHUB_API_TOKEN"] = api_token

In [6]:
from langchain.docstore.document import Document
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

In [7]:
documents = []
for file in uploaded_file:
    pdfReader = PdfReader(file)

    for i, page in enumerate(pdfReader.pages):
        documents.append(Document(page_content=page.extract_text(),
                                metadata={'title': file, 
                                            'page_number': i}))
chunks = get_chunks(documents, new_files=True)

In [17]:
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.vectorstores import Chroma
from langchain import HuggingFaceHub
import numpy as np
from langchain.chains import RetrievalQA

embeddings = HuggingFaceInstructEmbeddings(
        query_instruction="Represent the query for retrieval: ", 
        model_name=embeddings_model_name
        )

llm = HuggingFaceHub(repo_id=model_name)
db = Chroma.from_documents(chunks, embeddings, persist_directory='app_database/db', llm=llm)
retriever = db.as_retriever()
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)

No sentence-transformers model found with name C:\Users\abhis/.cache\torch\sentence_transformers\google_flan-t5-base. Creating a new one with MEAN pooling.
Some weights of the model checkpoint at C:\Users\abhis/.cache\torch\sentence_transformers\google_flan-t5-base were not used when initializing T5EncoderModel: ['lm_head.weight']
- This IS expected if you are initializing T5EncoderModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing T5EncoderModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Using embedded DuckDB with persistence: data will be stored in: app_database/db


In [18]:
initial_query = 'My name is Abhishek. What is the main idea of the paper?'

In [19]:
qa({'query': initial_query})

{'query': 'My name is Abhishek. What is the main idea of the paper?',
 'result': 'We use learned embeddings to convert the input tokens and output tokens to vector',
 'source_documents': [Document(page_content='size 1.\nThe dimensionality of input and output is dmodel = 512 , and the inner-layer has dimensionality\ndff= 2048 .\n3.4 Embeddings and Softmax\nSimilarly to other sequence transduction models, we use learned embeddings to convert the input\ntokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor-\nmation and softmax function to convert the decoder output to predicted next-token probabilities. In\nour model, we share the same weight matrix between the two embedding layers and the pre-softmax\nlinear transformation, similar to [ 30]. In the embedding layers, we multiply those weights bypdmodel.\n3.5 Positional Encoding\nSince our model contains no recurrence and no convolution, in order for the model to make use of the\norder of th

In [20]:
qa({'query': 'What is my name?'})

{'query': 'What is my name?',
 'result': 'i',
 'source_documents': [Document(page_content='input when generating the next.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n2', metadata={'title': 'data/1706.03762.pdf', 'page_number': 1}),
  Document(page_content='size 1.\nThe dimensionality of input and output is dmodel = 512 , and the inner-layer has dimensionality\ndff= 2048 .\n3.4 Embeddings and Softmax\nSimilarly to other sequence transduction models, we use learned embeddings to convert the input\ntokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor-\nmation and softmax function to convert the decoder output to predicted next-token probabilities. In\nour model, we share the same weight matrix between the two embedding layers and the pre-softmax\nlinear transformation

In [14]:
doc = [Document(page_content=initial_query, metadata={'title': 'query(ies)', 'query_number': 1})]
additional_chunks = get_chunks(doc, new_files=False)
query_ids = db.add_documents(additional_chunks)

In [15]:
qa({'query': 'What is my name?'})

{'ids': [['259af4e2-030a-11ee-9127-9bf0c229e25a',
   '2bdb759e-030a-11ee-8de1-9bf0c229e25a',
   'fdb10923-0309-11ee-82d1-9bf0c229e25a',
   'fdb10930-0309-11ee-86bf-9bf0c229e25a',
   'fdb1091a-0309-11ee-88a3-9bf0c229e25a',
   'fdb1091f-0309-11ee-8f95-9bf0c229e25a',
   'fdb10914-0309-11ee-ae26-9bf0c229e25a',
   'fdb10933-0309-11ee-86da-9bf0c229e25a',
   'fdb10936-0309-11ee-910f-9bf0c229e25a',
   'fdb10916-0309-11ee-9df5-9bf0c229e25a']],
 'embeddings': None,
 'documents': [['My name is Abhishek. What is the main idea of the paper?',
   'My name is Abhishek. What is the main idea of the paper?',
   'size 1.\nThe dimensionality of input and output is dmodel = 512 , and the inner-layer has dimensionality\ndff= 2048 .\n3.4 Embeddings and Softmax\nSimilarly to other sequence transduction models, we use learned embeddings to convert the input\ntokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor-\nmation and softmax function to convert the decod

In [ ]:
retriever = db.as_retriever()
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=True)
qa({'query': 'What is my name?'})

{'query': 'What is my name?',
 'result': 'Yvette',
 'source_documents': [Document(page_content='input when generating the next.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n2', metadata={'title': '1706.03762.pdf', 'page_number': 1}),
  Document(page_content='input when generating the next.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n2', metadata={'title': '1706.03762.pdf', 'page_number': 1}),
  Document(page_content='input when generating the next.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1

In [36]:
db._collection.get(ids = ['74cfd483-030a-11ee-973f-9bf0c229e25a',
  '74cfd4aa-030a-11ee-a7ee-9bf0c229e25a'])['documents']

['0', '1']

In [32]:
db.add_documents()

True

In [33]:
db._collection.delete(ids=['74cfd483-030a-11ee-973f-9bf0c229e25a',
  '74cfd4aa-030a-11ee-a7ee-9bf0c229e25a'])

[UUID('ae1618ab-8d41-4383-a662-7474c05ed516'),
 UUID('e2716b24-9886-441f-9a4a-bfadbd4c4be0')]

In [35]:
db.add_documents([Document(page_content=i, metadata={'title': 'query(ies)', 'query_number': 1}) for i in range(2)], ids = ['74cfd483-030a-11ee-973f-9bf0c229e25a',
  '74cfd4aa-030a-11ee-a7ee-9bf0c229e25a'])

# _collection.get(ids = ['74cfd483-030a-11ee-973f-9bf0c229e25a',
#   '74cfd4aa-030a-11ee-a7ee-9bf0c229e25a'])['documents']

['74cfd483-030a-11ee-973f-9bf0c229e25a',
 '74cfd4aa-030a-11ee-a7ee-9bf0c229e25a']

In [ ]:
random_string = """
.get also supports the where and where_document filters. If no ids are supplied, it will return all items in the collection that match the where and where_document filters.
When using get or query you can use the include parameter to specify which data you want returned - any of embeddings, documents, metadatas, and for query, distances. By default, Chroma will return the documents, metadatas and in the case of query, the distances of the results. embeddings are excluded by default for performance and the ids are always returned. You can specify which of these you want returned by passing an array of included field names to the includes parameter of the query or get method.
Chroma supports filtering queries by metadata and document contents. The where filter is used to filter by metadata, and the where_document filter is used to filter by document contents.
If an id is not found in the collection, an exception will be raised. If documents are supplied without corresponding embeddings, the embeddings will be recomupted with the collection's embedding function.
If the supplied embeddings are not the same dimension as the collection, an exception will be raised.
Chroma also supports an upsert operation, which updates existing items, or adds them if they don't yet exist.
"""

In [ ]:
qa({"query": query})

In [ ]:
# len(db.get()['ids'])
# db.get().keys() # ['ids', 'embeddings', 'documents', 'metadatas']
# db.get()['embeddings']
# for i in ['ids','documents', 'metadatas']:
#     print(f"{i} - {len(set(db.get()[i]))}")
# z = db._embedding_function.embed_documents(['z' for i in range(2000)])
db.aadd_documents([random_string])

['a0342c8c-0303-11ee-923b-9bf0c229e25a']

In [ ]:
# np.array(db._collection.peek(1)['embeddings']).shape
# len(db._collection.peek(1)['documents'])
db._collection.get(ids=['a0342c8c-0303-11ee-923b-9bf0c229e25a'])

{'ids': ['a0342c8c-0303-11ee-923b-9bf0c229e25a'],
 'embeddings': None,
 'documents': ["\n.get also supports the where and where_document filters. If no ids are supplied, it will return all items in the collection that match the where and where_document filters.\nWhen using get or query you can use the include parameter to specify which data you want returned - any of embeddings, documents, metadatas, and for query, distances. By default, Chroma will return the documents, metadatas and in the case of query, the distances of the results. embeddings are excluded by default for performance and the ids are always returned. You can specify which of these you want returned by passing an array of included field names to the includes parameter of the query or get method.\nChroma supports filtering queries by metadata and document contents. The where filter is used to filter by metadata, and the where_document filter is used to filter by document contents.\nIf an id is not found in the collectio

In [ ]:
# db.update_document('a0342c8c-0303-11ee-923b-9bf0c229e25a', )
# Document(page_content=page.extract_text(),
#                                 metadata={'title': file, 
#                                             'page_number': i})
# np.array(db._collection.peek(1)['embeddings']).shape
db._collection.peek(2)['metadatas']

[{'title': '1706.03762.pdf', 'page_number': 13},
 {'title': '1706.03762.pdf', 'page_number': 0}]

In [ ]:
template = """You are a chatbot having a conversation with a human.

Given the following extracted parts of a long document and a question, create a final answer.

{context}

{chat_history}
Human: {human_input}
Chatbot:"""

prompt = PromptTemplate(
    input_variables=["chat_history", "human_input", "context"], 
    template=template
)

memory = ConversationBufferMemory(memory_key="chat_history", input_key="human_input")



In [ ]:
prompt

PromptTemplate(input_variables=['chat_history', 'human_input', 'context'], output_parser=None, partial_variables={}, template='You are a chatbot having a conversation with a human.\n\nGiven the following extracted parts of a long document and a question, create a final answer.\n\n{context}\n\n{chat_history}\nHuman: {human_input}\nChatbot:', template_format='f-string', validate_template=True)

In [ ]:
from langchain.chains.llm import LLMChain
llm_chain = LLMChain(
        llm=chatbot.llm, prompt=prompt, verbose=True, memory=memory
    )

In [ ]:
llm_chain.predict(human_input='what is the document about?', context='')



> Entering new LLMChain chain...
Prompt after formatting:
You are a chatbot having a conversation with a human.

Given the following extracted parts of a long document and a question, create a final answer.




Human: what is the document about?
Chatbot:


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))